# 📰 Khobor (Readloop): Newspaper Article → Flashcard Pipeline
### Automated Exam Preparation for CDS & GATE-Aptitude using OCR, Gemini AI, Google Sheets & Anki

---

**Khobor** (Bengali/Assamese for *News*) is an end-to-end study pipeline tailored for engineering and defence exam aspirants. Photograph a physical newspaper column, upload it here, and Khobor will:
1. **Preprocess & Deskew**: Enhance contrast with CLAHE, rotate tilted scans, and check for blurriness using OpenCV.
2. **Extract Text**: Perform local OCR with Tesseract (with optional Google Cloud Vision fallback).
3. **AI Exam Analysis**: Segment articles, curate high-register vocabulary categorized by exam tier (CDS vs. GATE-aptitude), and pull testable current affairs facts using Google Gemini.
4. **Cloud Database**: Organize data across 3 relational tabs in Google Sheets (`Articles`, `Vocabulary`, `CurrentAffairs`).
5. **Spaced Repetition (SM-2)**: Attach SuperMemo-2 spaced-repetition schedules.
6. **Anki Deck Export**: Generate ready-to-import `.apkg` flashcards for offline review on Anki Mobile and Desktop.

```
[📷 Newspaper Photo] ➡️ [OpenCV Preprocess] ➡️ [Tesseract OCR] ➡️ [Gemini API]
                                                                    │
                     ┌─────────────────────────────────┴─────────────────────────────────┐
                     ▼                                                                 ▼
          [📊 Google Sheets Backend]                                       [🧠 SM-2 Spaced Repetition]
        (Articles / Vocab / Facts)                                                     │
                                                                                       ▼
                                                                           [📦 Anki .apkg Export]
                                                                           (Mobile & Desktop Review)
```

## 🛠️ Step 1: Install System & Python Dependencies
We need **Tesseract OCR** (Linux binary) and Python packages (`opencv-python`, `pytesseract`, `google-generativeai`, `gspread`, `genanki`).

In [ ]:
# Install system binary for Tesseract on Ubuntu (Google Colab container)
!apt-get update -qq && apt-get install -y -qq tesseract-ocr > /dev/null

# Install Python dependencies
!pip install -q opencv-python pytesseract google-generativeai gspread google-auth genanki tabulate

print("✅ System and Python dependencies installed successfully!")

## 🔑 Step 2: Authentication & Configuration
1. **Google Sheets Auth**: Uses Colab's native authentication popup.
2. **Gemini API Key**: Reads from Colab Secrets (`userdata.get('GEMINI_API_KEY')`) or prompts securely via `getpass`.

In [ ]:
import os
import getpass

# 1. Gemini API Key Configuration
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    if not GEMINI_API_KEY:
        raise ValueError("Key not found in Colab secrets")
    print("✅ Gemini API key loaded from Colab Secrets!")
except Exception:
    GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
    if not GEMINI_API_KEY:
        GEMINI_API_KEY = getpass.getpass("🔑 Enter your Gemini API Key (from https://aistudio.google.com/): ")
    print("✅ Gemini API key configured!")

# 2. Google Authentication for Google Sheets
try:
    from google.colab import auth
    print("🔐 Authenticating Google account for Google Sheets access...")
    auth.authenticate_user()
    print("✅ Google Authentication complete!")
except ImportError:
    print("ℹ️ Not in Google Colab; will use local/service account credentials if available.")

# Google Sheet name
SPREADSHEET_NAME = "Khobor_Exam_Prep"
print(f"📄 Target Google Sheet: '{SPREADSHEET_NAME}'")

## 📦 Step 3: Import Khobor Modules
We load the modular Python components: `ocr_module`, `gemini_extraction`, `sheets_storage`, `srs_engine`, and `anki_export`.

In [ ]:
import matplotlib.pyplot as plt
from tabulate import tabulate

from ocr_module import ImagePreprocessor, OCREngine
from gemini_extraction import GeminiExtractor
from sheets_storage import SheetsManager
from srs_engine import sm2_update, get_initial_srs_state
from anki_export import AnkiDeckExporter

# Initialize pipeline instances
ocr_engine = OCREngine()
gemini_extractor = GeminiExtractor(api_key=GEMINI_API_KEY, model_name="gemini-1.5-flash")
sheets_manager = SheetsManager(spreadsheet_name=SPREADSHEET_NAME)
anki_exporter = AnkiDeckExporter(registry_path="./anki_registry.json", deck_prefix="Readloop")

# Connect or create Google Sheet tabs
sheets_connected = sheets_manager.connect_or_create()
if sheets_connected:
    print(f"✅ Google Sheet is ready: {sheets_manager.get_sheet_url()}")
else:
    print("⚠️ Running in offline backup mode; will save data to local JSON backups.")

## 📸 Step 4: Upload Article Photograph (or Use Sample Demo Clipping)
You can upload a photo of a newspaper column from your smartphone or camera, or generate a realistic test clipping.

In [ ]:
import cv2
import numpy as np

def create_demo_clipping() -> str:
    """Generates a synthetic high-yield newspaper clipping for CDS/GATE demo testing."""
    demo_path = "sample_newspaper_article.png"
    # White canvas representing newspaper paper
    img = np.ones((700, 950, 3), dtype=np.uint8) * 248
    
    # Add title and article text lines
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_serif = cv2.FONT_HERSHEY_COMPLEX
    
    # Newspaper header
    cv2.putText(img, "THE NATIONAL CHRONICLE", (260, 45), font_serif, 0.9, (20, 20, 20), 2)
    cv2.line(img, (50, 60), (900, 60), (40, 40, 40), 2)
    
    # Article 1: Polity / Defence
    cv2.putText(img, "PARLIAMENT PASSES LANDMARK DEFENCE INDIGENISATION BILL", (60, 105), font, 0.65, (0, 0, 0), 2)
    lines_art1 = [
        "NEW DELHI: In an expeditious legislative move, the Parliament on Tuesday passed",
        "the Defence Procurement Modernisation Bill, 2026. The legislation mandates that at least",
        "65 percent of capital acquisitions for the Armed Forces must be sourced domestically.",
        "Defence experts hailed the decision as a decisive boost to indigenous self-reliance.",
        "Addressing the house, the Minister underscored the paramount need to curtail fiscal",
        "dependency on foreign defence contractors amidst escalating geopolitical friction.",
        "The bill establishes the National Defence Acquisition Council (NDAC), which will be",
        "presided over by the Chief of Defence Staff to expedite strategic acquisitions.",
        "Opposition members acknowledged the strategic utility but demanded laconic quarterly reports.",
    ]
    y = 145
    for line in lines_art1:
        cv2.putText(img, line, (60, y), font, 0.44, (30, 30, 30), 1)
        y += 32
        
    cv2.line(img, (50, y + 10), (900, y + 10), (160, 160, 160), 1)
    y += 45
    
    # Article 2: Tech / Economy
    cv2.putText(img, "RESERVE BANK ACCELERATES QUANTUM CRYPTOGRAPHY PILOT", (60, y), font, 0.65, (0, 0, 0), 2)
    lines_art2 = [
        "MUMBAI: The Reserve Bank of India announced a consortium of leading technical institutes",
        "to establish post-quantum encryption standards for the Unified Payments Interface (UPI).",
        "Financial cybersecurity remains vulnerable to impending quantum supremacy breakthroughs.",
        "The Governor emphasized that proactive deterrence is indispensable for monetary sovereignty.",
    ]
    y += 38
    for line in lines_art2:
        cv2.putText(img, line, (60, y), font, 0.44, (30, 30, 30), 1)
        y += 32

    # Save synthetic image
    cv2.imwrite(demo_path, img)
    return demo_path

# Choose upload or demo
print("Options:")
print("1. Upload your own newspaper photo")
print("2. Use generated sample newspaper clipping (Fast demo)")

image_path = None
try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        image_path = list(uploaded.keys())[0]
        print(f"✅ Uploaded file: {image_path}")
except Exception:
    pass

if not image_path:
    image_path = create_demo_clipping()
    print(f"✅ Created sample demonstration newspaper clipping: {image_path}")

## 🔎 Step 5: Image Preprocessing & OCR Extraction
Here we execute OpenCV preprocessing:
- **Grayscale Conversion**: Reduces color variations.
- **Blur Check**: Computes Laplacian variance $\text{Var}(\nabla^2 I)$ to detect blurry photos.
- **Contrast Enhancement**: Applies CLAHE (Contrast Limited Adaptive Histogram Equalization).
- **Deskewing**: Calculates text skew angle and straightens the page.
- **OCR Extraction**: Uses Tesseract OCR (with fallback to Cloud Vision if configured).

In [ ]:
# Run OCR & Preprocessing Pipeline
ocr_result = ocr_engine.process_article_image(
    image_path,
    enable_cloud_fallback=True,
    blur_threshold=80.0,
)

raw_text = ocr_result["raw_text"]
is_blurry = ocr_result["is_blurry"]
blur_score = ocr_result["blur_score"]
deskew_angle = ocr_result["deskew_angle"]
engine_used = ocr_result["engine_used"]

# Display Visual Preprocessing Comparison
original_bgr = cv2.imread(image_path)
original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
processed_gray = ocr_result["preprocessed_image"]

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.imshow(original_rgb)
plt.title(f"Original Image\n(Blur Variance: {blur_score:.1f} | Skew: {deskew_angle:.1f}°)", fontsize=11)
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(processed_gray, cmap="gray")
plt.title(f"Preprocessed (CLAHE + Deskewed)\nEngine: {engine_used.upper()}", fontsize=11)
plt.axis("off")
plt.tight_layout()
plt.show()

print("=" * 70)
print(f"OCR Status Summary:")
print(f"- Engine Used       : {engine_used}")
print(f"- Blurriness Check  : {'⚠️ Blurry' if is_blurry else '✅ Crisp / Clear'} (Score: {blur_score:.1f})")
print(f"- Deskew Correction : {deskew_angle:.2f}°")
print(f"- Extracted Words   : {ocr_result['word_count']}")
print("=" * 70)
print("\n--- [Raw OCR Text Preview] ---")
print(raw_text[:600] + ("..." if len(raw_text) > 600 else ""))

## 🧠 Step 6: Gemini Exam Analysis & Structured Extraction
The raw OCR text is sent to the Gemini API using the exact exam-prep system prompt:
- Auto-detects multi-article clippings and segments them into clean JSON.
- Tags vocabulary words with their target exam tier (**CDS**, **GATE-aptitude**, or **Both**).
- Identifies high-value testable facts for Current Affairs papers.
- On malformed JSON, automatically retries with follow-up validation.

In [ ]:
# Execute Gemini Extraction
print("🚀 Sending raw text to Gemini API for exam-focused information extraction...")
try:
    articles_extracted = gemini_extractor.extract(raw_text)
    print(f"✅ Extraction complete! Identified {len(articles_extracted)} distinct article(s).\n")
except Exception as e:
    print(f"❌ Gemini Extraction Error: {str(e)}")
    articles_extracted = []

# Display Extracted Content in Formatted Tables
for i, art in enumerate(articles_extracted, 1):
    print("═" * 80)
    print(f"ARTICLE #{i}: {art.get('headline', 'Untitled')}")
    print("═" * 80)
    print(f"📁 Category       : {art.get('category')}")
    print(f"🔍 OCR Confidence : {art.get('ocr_confidence')}")
    print(f"📝 Summary        : {art.get('summary')}\n")
    
    # Vocabulary Table
    vocab_rows = [
        [v.get("word"), v.get("exam_tier"), v.get("meaning"), v.get("usage_sentence")]
        for v in art.get("vocabulary", [])
    ]
    if vocab_rows:
        print("📖 HIGH-YIELD VOCABULARY:")
        print(tabulate(vocab_rows, headers=["Word", "Tier", "Meaning", "Example Usage"], tablefmt="grid"))
        print()
        
    # Current Affairs Facts Table
    facts_rows = [
        [f.get("fact"), f.get("likely_relevance")]
        for f in art.get("current_affairs_facts", [])
    ]
    if facts_rows:
        print("🌐 CURRENT AFFAIRS FACTS (CDS / GATE):")
        print(tabulate(facts_rows, headers=["Concrete Fact", "Likely Exam Relevance"], tablefmt="grid"))
        print()
        
    # Key Sentences
    if art.get("key_sentences"):
        print("✨ KEY SENTENCES:")
        for ks in art.get("key_sentences"):
            print(f"  • {ks}")
        print()

## 📊 Step 7: Save to Google Sheets Storage
Data is written to three tabs:
1. **`Articles`**: Headline, category, summary, key sentences, and raw OCR snippet.
2. **`Vocabulary`**: Word, definition, example, exam tier, linked `article_id`, and initial **SM-2 parameters** (`ease_factor=2.5`, `interval_days=1`, `repetitions=0`).
3. **`CurrentAffairs`**: Fact, exam relevance, category, linked `article_id`, and initial **SM-2 parameters**.

In [ ]:
# Write extracted records to Google Sheets
save_summary = sheets_manager.save_pipeline_output(
    articles=articles_extracted,
    raw_ocr_text=raw_text,
    backup_dir="./backups",
)

print("=" * 60)
print("💾 STORAGE SUMMARY:")
print(f"- Articles Saved    : {save_summary['articles_saved']}")
print(f"- Vocabulary Cards  : {save_summary['vocab_saved']}")
print(f"- Current Affairs   : {save_summary['facts_saved']}")

if save_summary["saved_to_sheets"]:
    print(f"\n✅ Successfully written to Google Sheets!")
    print(f"🔗 View Live Spreadsheet: {save_summary['sheet_url']}")
else:
    print(f"\nℹ️ Sheets write skipped / offline. Saved to local backup: {save_summary['backup_path']}")
print("=" * 60)

## 🃏 Step 8: SM-2 Spaced Repetition & Anki Deck Generation
We compile the extracted vocabulary and facts into two `.apkg` decks:
- **`Readloop - Vocabulary.apkg`**
- **`Readloop - Current Affairs.apkg`**

The built-in deduplication engine ensures that running this notebook multiple times on new articles only appends new cards without creating duplicates!

In [ ]:
# Gather cards from pipeline output
vocab_cards = save_summary["vocabulary_cards"]
fact_cards = save_summary["current_affairs_cards"]

# Export Anki decks
anki_summary = anki_exporter.export_decks(
    vocab_items=vocab_cards,
    fact_items=fact_cards,
    output_dir="./anki_decks",
    reset_registry=False,  # Set to True if you want a complete deck rebuild
)

print("=" * 60)
print("🎓 ANKI DECK GENERATION SUMMARY:")
print(f"- Vocabulary Deck      : {anki_summary['vocab_deck_path']}")
print(f"  • New Cards Added    : {anki_summary['vocab_cards_added']}")
print(f"  • Duplicates Skipped : {anki_summary['vocab_cards_skipped']}")
print(f"- Current Affairs Deck : {anki_summary['facts_deck_path']}")
print(f"  • New Cards Added    : {anki_summary['facts_cards_added']}")
print(f"  • Duplicates Skipped : {anki_summary['facts_cards_skipped']}")
print("=" * 60)

## 📥 Step 9: Download Anki Decks (.apkg)
Run the cell below to download your `.apkg` decks to your computer or phone.

### How to import into Anki:
1. **Anki Desktop (Windows / Mac / Linux)**: Double-click the downloaded `.apkg` file, or open Anki → *File* → *Import*.
2. **AnkiDroid (Android)**: Open AnkiDroid → Tap the 3 dots menu → *Import* → Select `.apkg`.
3. **AnkiMobile (iOS)**: AirDrop or open the file in the Files app and choose *Open in Anki*.

In [ ]:
try:
    from google.colab import files
    print("📥 Initiating browser download for Vocabulary Deck...")
    files.download(anki_summary["vocab_deck_path"])
    print("📥 Initiating browser download for Current Affairs Deck...")
    files.download(anki_summary["facts_deck_path"])
except Exception:
    print("ℹ️ If running locally, your .apkg decks are saved at:")
    print(f"  1. {os.path.abspath(anki_summary['vocab_deck_path'])}")
    print(f"  2. {os.path.abspath(anki_summary['facts_deck_path'])}")